In [ ]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_row", 160)

In [ ]:
# Set BASE to the station feature folder in the team handoff.
# If running in Colab, upload/copy the CSVs and change BASE accordingly.
BASE = Path(r"/content")

S2_FULL_CSV = BASE / "sentinel2_model_dataset_epo14_2562_2568_b100_watermask_v4_s2formula.csv"
S2_STRICT_CSV = BASE / "sentinel2_train_strict_candidates_epo14_2562_2568_b100_watermask_v4_s2formula.csv"
S2_RELAXED_CSV = BASE / "sentinel2_train_relaxed_candidates_epo14_2562_2568_b100_watermask_v4_s2formula.csv"
CONTEXT_CSV = BASE / "rswq_context_features_ctx_2562_2568_gsmap_era5h_strict_v2.csv"

OUT_DIR = BASE / "rswq_tabular_assembly_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR_LABEL = "2562_2568"

print("Output directory:", OUT_DIR)
for path in [S2_FULL_CSV, S2_STRICT_CSV, S2_RELAXED_CSV, CONTEXT_CSV]:
    print(path.name, "exists:", path.exists())


In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
def load_csv(path):
  if not path.exists():
    raise FileNotFoundError(path)
  return pd.read_csv(path)

s2_full = load_csv(S2_FULL_CSV)
s2_strict = load_csv(S2_STRICT_CSV)
s2_relaxed = load_csv(S2_RELAXED_CSV)
context = load_csv(CONTEXT_CSV)

print("s2_full:", s2_full.shape)
print("s2_strict:", s2_strict.shape)
print("s2_relaxed:", s2_relaxed.shape)
print("context:", context.shape)

In [ ]:
def check_sample_id(df, name):
    if "sample_id" not in df.columns:
        raise KeyError(f"{name} has no sample_id column")
    print(
        name,
        "rows:", len(df),
        "unique sample_id:", df["sample_id"].nunique(),
        "duplicates:", int(df["sample_id"].duplicated().sum()),
    )
    if df["sample_id"].duplicated().any():
        raise ValueError(f"{name} has duplicated sample_id")


check_sample_id(s2_full, "s2_full")
check_sample_id(s2_strict, "s2_strict")
check_sample_id(s2_relaxed, "s2_relaxed")
check_sample_id(context, "context")

full_ids = set(s2_full["sample_id"].astype(str))
context_ids = set(context["sample_id"].astype(str))

print("full/context sample_id intersection:", len(full_ids & context_ids))
print("full/context exact same set:", full_ids == context_ids)


In [ ]:
def merge_s2_with_context(s2_df, context_df):
    # Drop duplicate metadata columns from context to avoid _x/_y suffixes.
    # Keep the Sentinel-2/Ground Observation table as the main metadata source.
    duplicate_context_cols = sorted((set(s2_df.columns) & set(context_df.columns)) - {"sample_id"})
    context_to_add = context_df.drop(columns=duplicate_context_cols)

    print("Duplicate context metadata columns dropped:", duplicate_context_cols)
    print("Context columns added:", len(context_to_add.columns) - 1)

    merged = s2_df.merge(
        context_to_add,
        on="sample_id",
        how="left",
        validate="one_to_one",
    )
    return merged


In [ ]:
model_input_full = merge_s2_with_context(s2_full, context)
model_input_strict = merge_s2_with_context(s2_strict, context)
model_input_relaxed = merge_s2_with_context(s2_relaxed, context)

print("model_input_full:", model_input_full.shape)
print("model_input_strict:", model_input_strict.shape)
print("model_input_relaxed:", model_input_relaxed.shape)


In [ ]:
for name, df in {
    "full": model_input_full,
    "strict": model_input_strict,
    "relaxed": model_input_relaxed,
}.items():
    print("\n==", name, "==")
    print("shape:", df.shape)
    print("unique sample_id:", df["sample_id"].nunique())
    print("duplicate sample_id:", int(df["sample_id"].duplicated().sum()))
    if "match_window" in df.columns:
        print("match_window:", df["match_window"].value_counts(dropna=False).to_dict())
    if "data_quality" in df.columns:
        print("data_quality:", df["data_quality"].value_counts(dropna=False).to_dict())

display(model_input_full.head())


In [ ]:
full_out = OUT_DIR / f"rswq_model_input_full_s2_context_{YEAR_LABEL}.csv"
strict_out = OUT_DIR / f"rswq_model_input_strict_s2_context_{YEAR_LABEL}.csv"
relaxed_out = OUT_DIR / f"rswq_model_input_relaxed_s2_context_{YEAR_LABEL}.csv"

model_input_full.to_csv(full_out, index=False, encoding="utf-8-sig")
model_input_strict.to_csv(strict_out, index=False, encoding="utf-8-sig")
model_input_relaxed.to_csv(relaxed_out, index=False, encoding="utf-8-sig")

metadata = {
    "inputs": {
        "s2_full": str(S2_FULL_CSV),
        "s2_strict": str(S2_STRICT_CSV),
        "s2_relaxed": str(S2_RELAXED_CSV),
        "context": str(CONTEXT_CSV),
    },
    "outputs": {
        "full": str(full_out),
        "strict": str(strict_out),
        "relaxed": str(relaxed_out),
    },
    "merge_key": "sample_id",
    "note": "Simple assembly only. No split labels, flags, imputation, scaling, or extra feature engineering added.",
}

metadata_out = OUT_DIR / f"rswq_model_input_simple_assembly_metadata_{YEAR_LABEL}.json"
metadata_out.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")

for path in [full_out, strict_out, relaxed_out, metadata_out]:
    print(path, "size:", path.stat().st_size)
